In [ ]:
import os, json, numpy as np
from PIL import Image
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.amp import autocast, GradScaler
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, get_scheduler
from torch.utils.tensorboard import SummaryWriter
from peft import get_peft_model, LoraConfig, TaskType
from utils.my_tokenizer import Tokenizer
import utils.my_ecg_process as ecg
from utils.my_config import *

cut_12 = False

DATA_DIR_TRAIN = "data/qa_dataset/train"
DATA_DIR_VALID = "data/qa_dataset/valid"
DATA_DIR_VALID_OUTPUT = f"data/qa_dataset/valid_output"
IMAGE_ROOT = "data/img"
SIGNAL_ROOT = "data/records250/records250.npy"
IMG_CACHE_DIR = "siglip_cache" if cut_12 else "siglip_cache_no12cut"
HF_SIGLIP = "model/siglip-so400m-patch14-384"
HF_PHI3 = "model/phi-3"
OUTPUT_DIR = f"model/siglip_phi3_ecg"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

STAGE = 2
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
GRAD_CKPT = 10000
EPOCHS = 1
VISUAL_PREFIX_LEN = 2

os.makedirs(IMG_CACHE_DIR, exist_ok=True)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

In [ ]:
processed_data = np.load(processed_data_path)
sig_dataset = torch.tensor(processed_data, dtype=torch.float32)
sig_loader = DataLoader(sig_dataset, batch_size=32, shuffle=False)
beat = Tokenizer(**signal_cfg).to(DEVICE)
beat.eval()
checkpoint = torch.load(tokenizer_path, map_location=DEVICE, weights_only=True)
beat.load_state_dict(checkpoint['model_state_dict'], strict=False)

tokens = []
with torch.no_grad():
    for item in tqdm(sig_loader, desc="Processing Signals"):
        signal = item.to(DEVICE)
        _, _, _, _, _, quant = beat(signal[:, :, :seq_length], return_quant=True)
        tokens.extend(quant.detach().cpu())
        del signal, quant

In [ ]:
del beat
torch.cuda.empty_cache()
siglip = AutoModel.from_pretrained(HF_SIGLIP, trust_remote_code=False).to(DEVICE).eval()
for p in siglip.parameters():
    p.requires_grad = False

tokenizer_phi3 = AutoTokenizer.from_pretrained(HF_PHI3, trust_remote_code=False)
if tokenizer_phi3.pad_token is None:
    tokenizer_phi3.pad_token = tokenizer_phi3.eos_token
phi3 = AutoModelForCausalLM.from_pretrained(HF_PHI3, trust_remote_code=False).to(DEVICE)
phi3.gradient_checkpointing_enable()

special_tokens_dict = {
    "additional_special_tokens": [
        "<prediction>",
        "<image_start>", "<image_end>",
        "<signal_start>", "<signal_end>"
    ]
}
tokenizer_phi3.add_special_tokens(special_tokens_dict)
phi3.resize_token_embeddings(len(tokenizer_phi3))

phi3_hidden = phi3.config.hidden_size
siglip_dim = siglip.config.vision_config.hidden_size

image_start_id = tokenizer_phi3.convert_tokens_to_ids("<image_start>")
image_end_id = tokenizer_phi3.convert_tokens_to_ids("<image_end>")
signal_start_id = tokenizer_phi3.convert_tokens_to_ids("<signal_start>")
signal_end_id = tokenizer_phi3.convert_tokens_to_ids("<signal_end>")

os.makedirs(OUTPUT_DIR, exist_ok=True)
tokenizer_phi3.save_pretrained(OUTPUT_DIR)

phi3_embeddings = phi3.get_input_embeddings()
image_start_emb = phi3_embeddings(torch.tensor([image_start_id], device=DEVICE)).detach()
image_end_emb = phi3_embeddings(torch.tensor([image_end_id], device=DEVICE)).detach()
signal_start_emb = phi3_embeddings(torch.tensor([signal_start_id], device=DEVICE)).detach()
signal_end_emb = phi3_embeddings(torch.tensor([signal_end_id], device=DEVICE)).detach()

In [ ]:
def split_12leads(pil_img, lead_layout=(6,2)):
    w, h = pil_img.size
    rows, cols = lead_layout
    sub_w, sub_h = w // cols, h // rows
    leads = []
    for r in range(rows):
        for c in range(cols):
            crop = pil_img.crop((c*sub_w, r*sub_h, (c+1)*sub_w, (r+1)*sub_h))
            crop = crop.resize((384,384))
            leads.append(transform(crop))
    return torch.stack(leads, dim=0)

In [ ]:
import random

class ECGQADataset(Dataset):
    def __init__(self, qa_dir, stage=3, seed=42):
        self.samples = []
        for root, _, files in os.walk(qa_dir):
            for f in files:
                if f.endswith(".json"):
                    with open(os.path.join(root, f), "r", encoding="utf-8") as fp:
                        data = json.load(fp)
                        if isinstance(data, list):
                            self.samples.extend(data)
                        elif isinstance(data, dict) and "data" in data:
                            self.samples.extend(data["data"])
        rnd = random.Random(seed)
        rnd.shuffle(self.samples)
        if stage == 2:
            self.samples = rnd.sample(self.samples, k=int(len(self.samples)/5))
        print(f"[INFO] Found {len(self.samples)} QA samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [ ]:
signal_proj = nn.Sequential(nn.Linear(signal_dim, phi3_hidden), nn.GELU(), nn.LayerNorm(phi3_hidden)).to(DEVICE)
visual_proj = nn.Sequential(nn.Linear(siglip_dim, phi3_hidden*VISUAL_PREFIX_LEN),
                            nn.GELU(),
                            nn.Linear(phi3_hidden*VISUAL_PREFIX_LEN, phi3_hidden*VISUAL_PREFIX_LEN)).to(DEVICE)

peft_cfg = LoraConfig(r=64, lora_alpha=128, target_modules=["qkv_proj","o_proj"], lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM)
phi3 = get_peft_model(phi3, peft_cfg)

In [ ]:
def get_feats(item):
    filenames = item["filename"]
    img_feats, sig_feats = [], []
    for j, fn in enumerate(filenames):
        if item["label"][j] == "img":
            cache_path = os.path.join(IMG_CACHE_DIR, f"{fn.split('/')[-1]}.pt")
            if os.path.exists(cache_path):
                img_feat = torch.load(cache_path, weights_only=True)
            else:
                image = Image.open(os.path.join(IMAGE_ROOT, f"{fn}.png")).convert("RGB")
                if cut_12:
                    leads = split_12leads(image)  # [12,3,384,384]
                    with torch.no_grad(), autocast("cuda", dtype=torch.bfloat16):
                        vision_out = siglip.get_image_features(leads.to(DEVICE))
                    img_feat = vision_out.cpu()  # [12, feat_dim]
                    torch.save(img_feat.float(), cache_path)
                else:
                    image = transform(image).unsqueeze(0).to(DEVICE)
                    with torch.no_grad(), autocast("cuda", dtype=torch.bfloat16):
                        img_feat = siglip.get_image_features(image).view(1, -1).cpu()
                    torch.save(img_feat.float(), cache_path)
            img_feats.append(img_feat)
        else:
            sig_idx = int(fn.split('/')[-1]) - 1
            sig_feats.append(tokens[sig_idx].clone().detach())
    return img_feats, sig_feats

In [ ]:
def get_prompt(item):
    return f"<|user|> {item['question']} <|assistant|>"

In [ ]:
for stg in range(STAGE, 4):
    if stg == 2:
        seed = 40
        print("[INFO] Step 1: Modal pretraining (freeze LM)")
        for n, p in phi3.named_parameters():
            p.requires_grad = False
        params = [
            {"params": visual_proj.parameters(), "lr": 2e-4},
            {"params": signal_proj.parameters(), "lr": 2e-4},
        ]

    elif stg == 3:
        seed = 42
        print("[INFO] Step 2: Joint finetuning (unfreeze LM middle/high layers + LoRA)")
        for n, p in phi3.named_parameters():
            if "lora" in n:
                p.requires_grad = True
            else:
                p.requires_grad = False
        params = [
            {"params": visual_proj.parameters(), "lr": 2e-5},
            {"params": signal_proj.parameters(), "lr": 2e-5},
            {"params": [p for p in phi3.parameters() if p.requires_grad], "lr": 2e-5}
        ]
        
    train_ds = ECGQADataset(DATA_DIR_TRAIN, stage=stg, seed=seed)
    num_steps = len(train_ds)*EPOCHS // GRAD_ACCUM_STEPS
    optim = torch.optim.AdamW(params)
    scheduler = get_scheduler("linear", optim, num_warmup_steps=100, num_training_steps=num_steps)
    scaler = GradScaler()

    writer = SummaryWriter(log_dir=f"runs/phi3_siglip_beat_{stg}")
    phi3.train(); visual_proj.train(); signal_proj.train()
    optim.zero_grad(set_to_none=True)
    for epoch in range(EPOCHS):
        total_loss = 0
        for idx, item in enumerate(tqdm(train_ds, desc=f"{HF_PHI3.split('/')[-1]} STAGE {stg}", unit="sample")):
            # -----------------------------
            # IMAGE PROJECTION
            # -----------------------------
            img_feats, sig_feats = get_feats(item)

            with autocast("cuda", dtype=torch.bfloat16):
                # ---------------------------
                # IMAGE + SIGNAL PROJECTION
                # ---------------------------
                prefix_tokens = []
                if img_feats:
                    imgs = torch.stack(img_feats, dim=0).to(DEVICE)
                    if len(imgs.shape) > 3.5:
                        img = img.squeeze(0)
                    imgs.requires_grad_(True)
                    b, N, F = imgs.shape
                    img_proj_feats = visual_proj(imgs.view(b*N, F))
                    img_proj_feats = img_proj_feats.view(b, N*VISUAL_PREFIX_LEN, phi3_hidden)
                    prefix_tokens.extend([image_start_emb, img_proj_feats[0], image_end_emb])

                if sig_feats:
                    sig_tensor = torch.stack(sig_feats).to(DEVICE)
                    sig_tensor.requires_grad_(True)
                    sig_proj_feats = signal_proj(sig_tensor.float())
                    prefix_tokens.extend([signal_start_emb, sig_proj_feats[0], signal_end_emb])

                if prefix_tokens:
                    prefix_tokens = torch.cat(prefix_tokens, dim=0).unsqueeze(0)
                else:
                    prefix_tokens = None

                # ---------------------------
                # PROMPT EMBEDDING
                # ---------------------------
                prompt = get_prompt(item)
                answer = item.get("answer","")
                if isinstance(answer, list):
                    answer = ", ".join(answer).capitalize() + "."
                elif answer == "":
                    answer = "<prediction>"
                tok_q = tokenizer_phi3(prompt, truncation=True, padding=True, return_tensors="pt")
                tok_a = tokenizer_phi3(answer, truncation=True, padding=True, return_tensors="pt")
                text_emb = phi3.get_input_embeddings()(tok_q["input_ids"].to(DEVICE))
                ans_emb = phi3.get_input_embeddings()(tok_a["input_ids"].to(DEVICE))

                if prefix_tokens is not None and prefix_tokens.numel() > 0:
                    prefix_len = prefix_tokens.size(1)
                    input_embeds = torch.cat([prefix_tokens, text_emb, ans_emb], dim=1)
                    prefix_mask = torch.ones(1, prefix_len, device=DEVICE, dtype=torch.long)
                    attn_mask = torch.cat([
                        prefix_mask,
                        tok_q["attention_mask"].to(DEVICE).long(),
                        (tok_a["input_ids"].to(DEVICE) != tokenizer_phi3.pad_token_id).long()
                    ], dim=1)
                else:
                    prefix_len = 0
                    input_embeds = torch.cat([text_emb, ans_emb], dim=1)
                    attn_mask = torch.cat([
                        tok_q["attention_mask"].to(DEVICE).long(),
                        (tok_a["input_ids"].to(DEVICE) != tokenizer_phi3.pad_token_id).long()
                    ], dim=1)

                labels = tok_a["input_ids"].to(DEVICE).detach()
                labels[labels == tokenizer_phi3.pad_token_id] = -100
                labels_full = torch.cat([
                    torch.full((1, prefix_len + tok_q["input_ids"].size(1)), -100, device=DEVICE),
                    labels
                ], dim=1)
            
                # -----------------------------
                # BACKWARD
                # -----------------------------
                loss = phi3(
                    inputs_embeds=input_embeds,
                    attention_mask=attn_mask,
                    labels=labels_full,
                    use_cache=False
                ).loss

            scaler.scale(loss/GRAD_ACCUM_STEPS).backward()
            total_loss += loss.item()

            if (idx+1) % GRAD_ACCUM_STEPS == 0 or idx == len(train_ds)-1:
                writer.add_scalars('Loss', {'Train': loss.item()/GRAD_ACCUM_STEPS}, idx+1)
                has_any_grad = False
                for g in optim.param_groups:
                    for p in g['params']:
                        if p.grad is not None:
                            has_any_grad = True
                            break
                    if has_any_grad:
                        break

                if has_any_grad:
                    scaler.step(optim)
                    scaler.update()
                    scheduler.step()
                optim.zero_grad(set_to_none=True)

            if (idx+1) % GRAD_CKPT == 0:
                ckpt_num = (int(idx+1) / GRAD_CKPT) % 2
                ckpt_path = os.path.join(OUTPUT_DIR, f"ckpt_step_{ckpt_num}.pt")
                torch.save({
                    "phi3_state_dict": phi3.state_dict(),
                    "visual_proj_state_dict": visual_proj.state_dict(),
                    "signal_proj_state_dict": signal_proj.state_dict(),
                    "optim_state_dict": optim.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "step": idx+1,
                    "epoch": epoch+1
                }, ckpt_path)
                print(f"[INFO] Saved checkpoint: {ckpt_path}")

        print(f"Epoch {epoch+1}: Train Loss = {total_loss/len(train_ds):.4f}")
        save_dir = os.path.join(OUTPUT_DIR, f"stage_{stg}")
        phi3.save_pretrained(save_dir)
        torch.save(visual_proj.state_dict(), os.path.join(save_dir, "visual_proj.pt"))
        torch.save(signal_proj.state_dict(), os.path.join(save_dir, "signal_proj.pt"))
        print(f"[INFO] ✅ Model, tokenizer, and projections saved to {save_dir}")
        
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

In [ ]:
from pathlib import Path

os.makedirs(DATA_DIR_VALID_OUTPUT, exist_ok=True)
phi3.eval(); visual_proj.eval(); signal_proj.eval()
for json_path in list(Path(DATA_DIR_VALID).rglob("*.json")):
    json_path = str(json_path)
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    samples = data if isinstance(data, list) else data.get("data", [])

    json_path_new = json_path.replace(DATA_DIR_VALID, DATA_DIR_VALID_OUTPUT)
    dir_path = os.path.dirname(json_path_new)
    os.makedirs(dir_path, exist_ok=True)
    if os.path.exists(json_path_new):
        continue
    
    instruction = ""
    max_new_tokens = 40
    for item_idx, item in enumerate(tqdm(samples, desc=str(json_path))):
        img_feats, sig_feats = get_feats(item)

        with autocast("cuda", dtype=torch.bfloat16):
            # ---------------------------
            # IMAGE + SIGNAL PROJECTION
            # ---------------------------
            prefix_tokens = []
            if img_feats:
                imgs = torch.stack(img_feats, dim=0).to(DEVICE)
                if len(imgs.shape) > 3.5:
                    img = img.squeeze(0)
                b, N, F = imgs.shape
                img_proj_feats = visual_proj(imgs.view(b*N, F))
                img_proj_feats = img_proj_feats.view(b, N*VISUAL_PREFIX_LEN, phi3_hidden)
                prefix_tokens.extend([image_start_emb, img_proj_feats[0], image_end_emb])

            if sig_feats:
                sig_tensor = torch.stack(sig_feats).to(DEVICE)
                sig_proj_feats = signal_proj(sig_tensor.float())
                prefix_tokens.extend([signal_start_emb, sig_proj_feats[0], signal_end_emb])

            if prefix_tokens:
                prefix_tokens = torch.cat(prefix_tokens, dim=0).unsqueeze(0)
            else:
                prefix_tokens = None

            # ---------------------------
            # PROMPT EMBEDDING
            # ---------------------------
            prompt = get_prompt(item)
            tok_q = tokenizer_phi3(prompt, truncation=True, padding=True, return_tensors="pt")
            text_emb = phi3.get_input_embeddings()(tok_q["input_ids"].to(DEVICE))
            if prefix_tokens is not None and prefix_tokens.numel() > 0:
                input_embeds = torch.cat([prefix_tokens, text_emb], dim=1)
                attn_mask = torch.cat([torch.ones(1, prefix_tokens.size(1), device=DEVICE), tok_q["attention_mask"].to(DEVICE)], dim=1)
            else:
                input_embeds = torch.cat([text_emb], dim=1)
                attn_mask = torch.cat([tok_q["attention_mask"].to(DEVICE)], dim=1)

            # ---------------------------
            # GENERATE
            # ---------------------------
            with torch.no_grad():
                generated = phi3.generate(
                    inputs_embeds=input_embeds,
                    attention_mask=attn_mask,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    repetition_penalty=1.1,
                    pad_token_id=tokenizer_phi3.pad_token_id,
                    eos_token_id=tokenizer_phi3.eos_token_id,
                    use_cache=True
                )
            decoded = tokenizer_phi3.batch_decode(generated, skip_special_tokens=True)[0]
            item["gen_answer"] = decoded

    with open(json_path_new, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)